# Coarse-grained RNA simulations with CALVADOS — a hands-on tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshuJS/V21-RNA-simulation-CALVADOS-tutorial/blob/main/V21_RNA_CALVADOS_tutorial.ipynb)

Learn **molecular dynamics (MD)** and **coarse-grained (CG)** modelling of RNA through one example: a 21-nucleotide RNA, **V21** (`UAUAAUUAAUAACUAAUUACU`). You'll learn the ideas (with equations + live animations), **set up** a CALVADOS simulation while **visualising it in 3-D** at each step, and **analyse** pre-computed trajectories (shipped here) — shape, salt response, and contact maps.

> Runs on a **free Colab CPU runtime** — run the cells top to bottom.


## ▶ Step 0 — Run this cell first (downloads data + installs)

This single cell downloads the tutorial data and installs the few libraries we need. **Run it before any other cell.** It always re-downloads a fresh copy of the data, so if you ever hit a `FileNotFoundError` later, just come back and run this cell again. (We don't run MD in the notebook — all analysis is plain **NumPy**, no compiled trajectory reader, so no version-compatibility headaches.)

In [ ]:
import sys, subprocess, os, importlib, importlib.util, pathlib, shutil, numpy as np

# 1) DATA — always re-clone fresh, so re-running this cell fixes any stale/old copy
REPO = "V21-RNA-simulation-CALVADOS-tutorial"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1",
                f"https://github.com/anshuJS/{REPO}.git"], check=True)
DATA = f"{REPO}/data"
assert os.path.exists(f"{DATA}/V21_top.pdb"), "data download failed — run this cell again"

# 2) install the few libraries we need (3-D viewer + CALVADOS config module)
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("py3Dmol", "Jinja2", "pyyaml")
pip("--no-deps", "git+https://github.com/KULL-Centre/CALVADOS.git@v0.7.0")

# CALVADOS __init__ eagerly imports a heavy analysis stack (MDAnalysis, numba, localcider, ...)
# that doesn't install on Colab; we only use the self-contained calvados.cfg, so trim __init__.
importlib.invalidate_caches()
pkg = pathlib.Path(importlib.util.find_spec("calvados").origin).parent
(pkg / "__init__.py").write_text("# trimmed for this tutorial: only calvados.cfg is used\n")
from calvados.cfg import Config, Components
import py3Dmol
print("✅ setup complete — numpy", np.__version__, "| data:", sorted(os.listdir(DATA)))

### Lightweight helpers (NumPy + py3Dmol)

A few small functions: load a trajectory (NumPy arrays of coordinates + box), unwrap the chain across periodic boundaries, collapse the 2 beads/nucleotide into per-residue centroids, and turn a single frame into a PDB string for 3-D viewing. No `mdtraj` — just NumPy.

In [ ]:
TOP_LINES = [l.rstrip("\n") for l in open(f"{DATA}/V21_top.pdb") if l.startswith(("ATOM","HETATM"))]

def load_coords(variant, pre, salt):
    z = np.load(f"{DATA}/trajectories/{variant}/V21_{pre}_{salt}mM/coords.npz")
    return z["xyz"].astype(np.float64), z["box"].astype(np.float64)        # nm

def _unwrap(xyz, box):                                  # follow the chain via minimum image
    d = xyz[..., 1:, :] - xyz[..., :-1, :]
    d -= np.round(d / box[..., None, :]) * box[..., None, :]
    uw = np.empty_like(xyz); uw[..., 0, :] = xyz[..., 0, :]
    uw[..., 1:, :] = xyz[..., :1, :] + np.cumsum(d, axis=-2)
    return uw

def residue_centroids(xyz, box):                        # (F,42,3) -> (F,21,3)
    uw = _unwrap(xyz, box)
    return uw.reshape(xyz.shape[0], -1, 2, 3).mean(2)

def frame_pdb(coords_nm):                               # one frame (42,3 nm) -> PDB text
    out = ["MODEL"]
    for line, (x, y, z) in zip(TOP_LINES, coords_nm * 10.0):              # nm -> Angstrom
        out.append(f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}")
    out.append("ENDMDL"); return "\n".join(out) + "\n"

def show_beads(variant, pre, salt, frame=0, pairs=None):
    xyz, box = load_coords(variant, pre, salt)
    fr = _unwrap(xyz[frame], box[frame])
    v = py3Dmol.view(width=520, height=420); v.addModel(frame_pdb(fr), "pdb")
    v.setStyle({"elem":"P"}, {"sphere":{"color":"orange","radius":3.2}})       # phosphate beads
    v.setStyle({"elem":"N"}, {"sphere":{"color":"forestgreen","radius":3.2}})  # base beads
    if pairs:
        a = fr * 10.0
        for i, j in pairs:                              # restraint between base beads (res p -> bead 2p-1, 0-based)
            p, q = a[2*i-1], a[2*j-1]
            v.addCylinder({"start":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                           "end":{"x":float(q[0]),"y":float(q[1]),"z":float(q[2])},
                           "radius":0.6,"color":"red","dashed":True})
    v.zoomTo(); return v.show()

def animate_traj(variant, pre, salt, nframes=60):
    """animate the uploaded trajectory in 3-D (py3Dmol)."""
    xyz, box = load_coords(variant, pre, salt)
    idx = np.linspace(0, len(xyz)-1, nframes).astype(int)
    blocks = []
    for fi in idx:
        fr = _unwrap(xyz[fi], box[fi]); fr = (fr - fr.mean(0)) + 2.0
        blocks.append("MODEL")
        for line, (x, y, z) in zip(TOP_LINES, fr*10.0):
            blocks.append(f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}")
        blocks.append("ENDMDL")
    v = py3Dmol.view(width=520, height=420)
    v.addModelsAsFrames("\n".join(blocks) + "\n", "pdb")
    v.setStyle({"elem":"P"}, {"sphere":{"color":"orange","radius":3.2}})
    v.setStyle({"elem":"N"}, {"sphere":{"color":"forestgreen","radius":3.2}})
    v.zoomTo(); v.animate({"loop":"forward","interval":80}); return v.show()
print("helpers ready")

---
# Part 1 — The ideas

## 1.1 What is a molecular dynamics simulation?

MD integrates **Newton's equations** for every particle; the force is the negative gradient of a potential energy $U$ (the *force field*):

$$\mathbf{F}_i = m_i\,\mathbf{a}_i = -\nabla_i\, U,\qquad
\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}\,\Delta t + \tfrac{1}{2}\mathbf{a}\,\Delta t^2$$

Repeating this tiny step millions of times produces a **trajectory** — a movie of the molecule — from which every physical property is a statistical average. The simplest case: one bead in a **harmonic well** (exactly what a *bond* does).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
k = 5.0; xs = np.linspace(-2, 2, 300); U = 0.5*k*xs**2
fig, ax = plt.subplots(figsize=(5.5, 3.2)); ax.plot(xs, U, color="steelblue", lw=2)
ax.set_xlabel("bead position $x$"); ax.set_ylabel("$U(x)$")
ax.set_title("a bead oscillating in a harmonic well  ($F=-dU/dx$)")
bead, = ax.plot([], [], "o", color="crimson", ms=16); A, w = 1.6, 2.2
def upd(i):
    xb = A*np.cos(w*i*0.08); bead.set_data([xb], [0.5*k*xb**2]); return (bead,)
anim = animation.FuncAnimation(fig, upd, frames=70, interval=70, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

## A more realistic interaction: the Lennard-Jones potential

Two beads that are *not* bonded can't overlap (hard repulsion) yet weakly attract at medium range (van der Waals). The **Lennard-Jones** potential captures both:

$$U_\text{LJ}(r)=4\varepsilon\left[\left(\tfrac{\sigma}{r}\right)^{12}-\left(\tfrac{\sigma}{r}\right)^{6}\right]$$

The $r^{-12}$ term is the repulsive **wall** (excluded volume, size $\sigma$); the $r^{-6}$ term is the attractive tail; they balance at $r=2^{1/6}\sigma$ with depth $\varepsilon$. This is the backbone of CALVADOS's **Ashbaugh–Hatch** term (1.4). Watch a free bead fall in and settle.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
sig, eps = 0.62, 1.0; rmin = 2**(1/6)*sig
rr = np.linspace(0.55, 2.2, 400); Ulj = 4*eps*((sig/rr)**12 - (sig/rr)**6)
def Flj(r):
    f = 24*eps*(2*(sig/r)**12 - (sig/r)**6)/r; return max(min(f, 120.0), -120.0)
pos, vel, dt, gamma, traj = 2.0*sig, -1.5, 1.2e-3, 1.8, []
for _ in range(1800):
    a = Flj(pos) - gamma*vel; vel += a*dt; pos += vel*dt
    pos = min(max(pos, 0.92*sig), 2.3); traj.append(pos)
traj = traj[::20]
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4), gridspec_kw={"width_ratios":[1.5,1]})
ax[0].plot(rr, Ulj, color="teal", lw=2); ax[0].axhline(0, color="gray", lw=0.6)
ax[0].axvline(rmin, color="gray", ls=":", lw=1); ax[0].set_ylim(-1.5, 3)
ax[0].set_xlabel("distance $r$ (nm)"); ax[0].set_ylabel("$U_{LJ}$"); ax[0].set_title("Lennard-Jones")
dot, = ax[0].plot([], [], "o", color="crimson", ms=12)
ax[1].set_xlim(-0.2, 2.3); ax[1].set_ylim(-1, 1); ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_title("two non-bonded beads"); ax[1].plot([0], [0], "o", color="royalblue", ms=26)
b1, = ax[1].plot([], [], "o", color="crimson", ms=26)
def upd(i):
    r = traj[i % len(traj)]; dot.set_data([r], [4*eps*((sig/r)**12-(sig/r)**6)]); b1.set_data([r], [0])
    return dot, b1
anim = animation.FuncAnimation(fig, upd, frames=len(traj), interval=55, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

## Cutoffs: short-range vs long-range forces

Evaluating all $N^2$ pairs every step is expensive, so we only compute interactions within a **cutoff** $r_\text{cut}$ and treat the rest as zero. Safe for **short-range** potentials (LJ $\sim r^{-6}$ → ~zero past 2–3 $\sigma$), but **not** for **long-range** electrostatics ($\sim 1/r$). The sweep shows how much of each potential a given cutoff captures.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
sig, eps, lB = 0.62, 1.0, 0.71
r = np.linspace(0.55, 5.0, 600); U_s = 4*eps*((sig/r)**12-(sig/r)**6); U_l = lB/r
fig, ax = plt.subplots(figsize=(6.8, 3.7))
ax.plot(r, U_s, color="teal", lw=1, alpha=0.3); ax.plot(r, U_l, color="purple", lw=1, alpha=0.3)
ls, = ax.plot([], [], color="teal", lw=2.6, label="LJ (short-range)")
ll, = ax.plot([], [], color="purple", lw=2.6, label="electrostatic (long-range)")
vline = ax.axvline(0, color="red", ls="--", lw=1.5); ax.axhline(0, color="gray", lw=0.5)
ax.set_ylim(-1.2, 2.6); ax.set_xlabel("distance $r$ (nm)"); ax.set_ylabel("$U$"); ax.legend(loc="upper right", fontsize=8)
txt = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)
ts, tl = np.trapz(np.abs(U_s), r), np.trapz(np.abs(U_l), r); cuts = np.linspace(0.9, 4.6, 48)
def upd(i):
    rc = cuts[i % len(cuts)]; m = r <= rc
    ls.set_data(r[m], U_s[m]); ll.set_data(r[m], U_l[m]); vline.set_xdata([rc, rc])
    txt.set_text(f"cutoff = {rc:.1f} nm\nLJ captured: {np.trapz(np.abs(U_s[m]),r[m])/ts*100:3.0f}%\n"
                 f"electrostatic captured: {np.trapz(np.abs(U_l[m]),r[m])/tl*100:3.0f}%")
    ax.set_title("beyond the cutoff, the interaction is set to 0"); return ls, ll, vline, txt
anim = animation.FuncAnimation(fig, upd, frames=len(cuts), interval=110, blit=False)
plt.close(fig); HTML(anim.to_jshtml())

## 1.2 Why coarse-graining? — atoms become beads

A fully **atomistic** model tracks every atom (and water): accurate, but motions take micro- to milliseconds while the timestep is ~1 fs — billions of steps. **Coarse-graining** groups atoms into a few **beads**, cutting particle count, allowing a bigger timestep, and using an *implicit* solvent.

| | all-atom | coarse-grained (CALVADOS) |
|---|---|---|
| RNA nucleotide | ~30 atoms | **2 beads** (phosphate + base) |
| solvent | explicit water | implicit (Debye–Hückel) |
| timestep | ~1–2 fs | ~10 fs |

Below is a **real all-atom RNA** (5 nucleotides, sticks + atoms) with the **CALVADOS beads overlaid** at their mapped positions — orange at each phosphate, green at each base. You can see ~30 atoms per nucleotide collapse into just **2 beads**.

In [ ]:
import numpy as np, py3Dmol, collections
aa = open(f"{DATA}/example_rna_allatom.pdb").read()
BASE_RING = {"N1","C2","N3","C4","C5","C6","N7","C8","N9"}
byres = collections.defaultdict(list)
for l in aa.splitlines():
    if l.startswith(("ATOM","HETATM")):
        byres[int(l[22:26])].append((l[12:16].strip(),
            (float(l[30:38]), float(l[38:46]), float(l[46:54]))))
beads = []
for rs, atoms in byres.items():
    P = [x for n, x in atoms if n == "P"]
    base = [x for n, x in atoms if n in BASE_RING]
    if P:    beads.append(("orange", P[0]))
    if base: beads.append(("forestgreen", tuple(np.mean(base, 0))))
v = py3Dmol.view(width=600, height=440)
v.addModel(aa, "pdb")
v.setStyle({"stick":{"radius":0.13}, "sphere":{"scale":0.22}})       # real molecule: atoms + bonds
for color, (x, y, z) in beads:                                       # CG beads overlaid, translucent
    v.addSphere({"center":{"x":x,"y":y,"z":z}, "radius":1.8, "color":color, "opacity":0.55})
v.zoomTo(); v.show()

## 1.3–1.4 The CALVADOS force field

Each bead has a size $\sigma$, charge $q$, and stickiness $\lambda$. The energy is a sum of simple terms:

$$U = U_\text{bond}+U_\text{angle} + U_\text{AH} + U_\text{DH}$$

The **Ashbaugh–Hatch** term is the LJ above with a knob $\lambda$ scaling the attraction (high $\lambda$ = sticky/collapsed, low = expanded). The **Debye–Hückel** term is screened electrostatics — and **salt screens charge**, which is exactly what compacts the RNA in Part 3.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
lB = 0.71; r = np.linspace(0.6, 6.0, 400)
def kappa(I): return np.sqrt(8*np.pi*lB*I*6.022e23/1e24) if I > 0 else 1e-6
salts = [0,10,20,50,100,150,300,500,1000]
fig, ax = plt.subplots(figsize=(5.6, 3.4)); line, = ax.plot([], [], lw=2, color="purple")
ax.set_xlim(0.6, 6); ax.set_ylim(0, 1.2); ax.set_xlabel("phosphate–phosphate $r$ (nm)"); ax.set_ylabel("$U_{DH}$ ($k_BT$)")
txt = ax.text(0.97, 0.9, "", transform=ax.transAxes, ha="right")
def upd(i):
    s = salts[i % len(salts)]; line.set_data(r, lB*np.exp(-kappa(s/1000)*r)/r)
    txt.set_text(f"salt = {s} mM"); ax.set_title("Debye–Hückel: salt screens the charge"); return line, txt
anim = animation.FuncAnimation(fig, upd, frames=len(salts), interval=600, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

---
# Part 2 — Setting up the V21 simulation (with 3-D visualisation)

We configure V21 the way CALVADOS expects, looking at what we build. (We *configure* only — a long run needs a GPU.)

V21 = 21 nucleotides → **42 beads**. Two beads/residue (phosphate, base), so residue $p$ (1-based) has base bead $2p$ — the index we restrain for the hairpin. CALVADOS has **no C/G base parameters** (base identity only sets bead size), so we use the **generic `r` bead** for all 21.

In [ ]:
SEQ        = "UAUAAUUAAUAACUAAUUACU"
NRES       = len(SEQ)
STEM_PAIRS = [(3,19),(4,18),(5,17),(6,16),(7,15),(8,14)]   # RNAfold hairpin stem
base_bead  = lambda p: 2*p
os.makedirs("V21_setup", exist_ok=True)
open("V21_setup/residues_C2RNA.csv","w").write(open(f"{DATA}/residues_C2RNA.csv").read())
open("V21_setup/rna.fasta","w").write(">V21\n" + "r"*NRES + "\n")
print(f"{SEQ} -> {NRES} nt = {2*NRES} beads; stem base-bead pairs:",
      [(base_bead(i), base_bead(j)) for i,j in STEM_PAIRS])

### Write `prepare.py` and generate inputs

A small `prepare.py` defines a `Config` + `Components` and writes `config.yaml`, `components.yaml`, `run.py`. Salt enters as `ionic` (molar); we set 150 mM and `platform='CPU'`. The hairpin adds 6 harmonic **custom restraints** (`cres`) between the stem base beads.

In [ ]:
def write_setup(folder, hairpin):
    os.makedirs(f"{folder}/input", exist_ok=True)
    open(f"{folder}/residues_C2RNA.csv","w").write(open(f"{DATA}/residues_C2RNA.csv").read())
    open(f"{folder}/rna.fasta","w").write(">V21\n" + "r"*NRES + "\n")
    cres = ""
    if hairpin:
        with open(f"{folder}/input/cres.txt","w") as f:
            for i,j in STEM_PAIRS:
                f.write(f"V21 1 {base_bead(i)} | V21 1 {base_bead(j)} | 0.6238 700.0\n")
        cres = ("\n    custom_restraints=True,\n    custom_restraint_type='harmonic',"
                "\n    fcustom_restraints=f'{cwd}/input/cres.txt',")
    prep = f"""
import os, subprocess
from calvados.cfg import Config, Components
cwd=os.getcwd(); sysname='V21'
config=Config(sysname=sysname, box=[40,40,40], temp=300, ionic=0.15, pH=7.0, topol='center',
    wfreq=1000, steps=2000, runtime=0, platform='CPU', restart='checkpoint',
    frestart='restart.chk', verbose=True{cres})
path=f'{{cwd}}/{{sysname}}'; subprocess.run(f'mkdir -p {{path}}',shell=True); subprocess.run('mkdir -p data',shell=True)
config.write(path, name='config.yaml')
c=Components(); c.add(name='V21', molecule_type='rna', restraint=False, charge_termini='both',
    fresidues=f'{{cwd}}/residues_C2RNA.csv', ffasta=f'{{cwd}}/rna.fasta', nmol=1,
    rna_kb1=1400.0, rna_kb2=2200.0, rna_ka=4.20, rna_pa=3.14,
    rna_nb_sigma=0.4, rna_nb_scale=15, rna_nb_cutoff=2.0)
c.write(path, name='components.yaml')
"""
    open(f"{folder}/prepare.py","w").write(prep)
    r = subprocess.run([sys.executable,"prepare.py"], cwd=folder, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
    return r.returncode

print("disordered ->", write_setup("V21_dis", hairpin=False), "| hairpin ->", write_setup("V21_hp", hairpin=True))
print("hairpin restraints (cres.txt):\n" + open("V21_hp/input/cres.txt").read())

### Look at it: the 42-bead V21 chain, and the hairpin restraints

Left/first: a real CALVADOS snapshot of V21 (orange = phosphate, green = base). For the hairpin we draw the 6 stem restraints as red dashed bonds on a *folded* snapshot — you can see them holding the stem closed.

In [ ]:
from IPython.display import display
print("V21 — extended (disordered, 0 mM):")
display(show_beads("disordered", "dis", 0, frame=0))
print("V21 — hairpin, with the 6 stem restraints (red dashed):")
display(show_beads("hairpin", "hp", 150, frame=500, pairs=STEM_PAIRS))

---
# ▶ Optional — run a short MD simulation yourself (CPU)

So far we *configured* a run. Here you actually **run molecular dynamics** — and the best part: we build the force field **directly in OpenMM**, so the Part-1 equations become code. Each bead gets a mass; we add **harmonic bonds** + an **angle** term (the backbone), the **Ashbaugh–Hatch** nonbonded term, and the **Debye–Hückel** electrostatics (with the salt-dependent screening $\kappa$). Then we integrate with a Langevin thermostat for a few thousand steps on the CPU and watch $R_g$ evolve.

> This is a **compact teaching implementation** of a CALVADOS-*style* CG RNA — enough to feel MD run and see salt compact the chain. The shipped trajectories below used the **full CALVADOS** model (and a GPU for 1 µs); those are what we analyse quantitatively.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm"], check=True)
import openmm as mm, openmm.unit as uu, numpy as np, matplotlib.pyplot as plt

def build_and_run(salt_mM, lam=0.2, nsteps=12000, nout=30):
    xyz, box = load_coords("disordered", "dis", 150)             # start from a shipped snapshot
    fr = _unwrap(xyz[0], box[0]); fr = fr - fr.mean(0) + 20.0
    nres = 21; s = mm.System()
    for k in range(nres): s.addParticle(194.1); s.addParticle(126.3)        # phosphate, base masses
    hb = mm.HarmonicBondForce(); ex = []                                    # backbone bonds
    for k in range(nres):
        p, b = 2*k, 2*k+1; hb.addBond(p, b, 0.40, 2200.); ex.append((p, b))
        if k < nres-1: hb.addBond(p, p+2, 0.50, 1400.); ex.append((p, p+2))
    s.addForce(hb)
    ha = mm.HarmonicAngleForce()                                            # backbone stiffness
    for k in range(nres-2): ha.addAngle(2*k, 2*k+2, 2*k+4, 3.14, 4.2)
    s.addForce(ha)
    ah = mm.CustomNonbondedForce("select(step(rm-r), lj+(1-l)*eps, l*lj); lj=4*eps*((sg/r)^12-(sg/r)^6);"
                                 "rm=1.122462*sg; sg=0.5*(s1+s2); l=0.5*(l1+l2)")              # Ashbaugh-Hatch
    ah.addGlobalParameter("eps", 0.8368); ah.addPerParticleParameter("s"); ah.addPerParticleParameter("l")
    kappa = np.sqrt(8*np.pi*0.71*(salt_mM/1000)*6.022e23/1e24) if salt_mM > 0 else 1e-6           # Debye screening
    dh = mm.CustomNonbondedForce("138.935*q1*q2*exp(-kk*r)/(80*r)")                                # Debye-Huckel
    dh.addGlobalParameter("kk", kappa); dh.addPerParticleParameter("q")
    for k in range(nres):
        ah.addParticle([0.6954, lam]); ah.addParticle([0.6238, lam]); dh.addParticle([-1.0]); dh.addParticle([0.0])
    for f in (ah, dh):
        f.setNonbondedMethod(mm.CustomNonbondedForce.CutoffNonPeriodic); f.setCutoffDistance(2.5)
        for i, j in ex: f.addExclusion(i, j)
        s.addForce(f)
    ig = mm.LangevinMiddleIntegrator(300*uu.kelvin, 0.01/uu.picosecond, 0.01*uu.picosecond)
    c = mm.Context(s, ig, mm.Platform.getPlatformByName("CPU")); c.setPositions(fr*uu.nanometer)
    mm.LocalEnergyMinimizer.minimize(c, maxIterations=200)
    rg, dt_ns = [], (nsteps//nout)*0.01/1000
    for _ in range(nout):
        ig.step(nsteps//nout)
        p = c.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(uu.nanometer)
        cc = p.reshape(nres, 2, 3).mean(1); rg.append(float(np.sqrt(((cc-cc.mean(0))**2).sum(-1).mean())))
    return np.arange(nout)*dt_ns, rg

print("running two short CPU simulations from the same start (low vs high salt)...")
t, rg_lo = build_and_run(0); _, rg_hi = build_and_run(1000)
plt.figure(figsize=(6.4, 3.7))
plt.plot(t, rg_lo, "-o", ms=3, color="#4575b4", label="0 mM (low salt)")
plt.plot(t, rg_hi, "-o", ms=3, color="#d73027", label="1000 mM (high salt)")
plt.xlabel("time (ns)"); plt.ylabel("$R_g$ (nm)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("your own MD: Debye screening at high salt compacts the RNA"); plt.tight_layout(); plt.show()
print(f"mean Rg:  0 mM = {np.mean(rg_lo[10:]):.2f} nm   1000 mM = {np.mean(rg_hi[10:]):.2f} nm")

---
# Part 3 — Analysing pre-simulated trajectories

The repo ships **1 µs CALVADOS runs of V21 at 12 salts** (0–1000 mM), disordered + hairpin (down-sampled to 1000 frames). Everything below is plain NumPy on the shipped coordinate arrays. We'll measure two classic shape descriptors — the **radius of gyration** and the **end-to-end distance**. First: what *is* the radius of gyration?

## What is the radius of gyration? (and the gyration tensor)

$R_g$ is the most-used measure of a molecule's overall size — the root-mean-square distance of its beads from their centre of mass:

$$R_g=\sqrt{\tfrac{1}{N}\sum_i\lVert\mathbf r_i-\mathbf r_\text{com}\rVert^2}$$

Small $R_g$ = compact; large $R_g$ = extended.

**The tensor view — more than just a size.** That sum can be packaged into the **gyration tensor**, a symmetric $3\times3$ matrix built from the *outer products* of the displacement vectors $\Delta\mathbf r_i=\mathbf r_i-\mathbf r_\text{com}$:

$$S=\frac1N\sum_i \Delta\mathbf r_i\,\Delta\mathbf r_i^{\mathsf T}\,,\qquad S_{ab}=\frac1N\sum_i \Delta r_{i,a}\,\Delta r_{i,b}\quad(a,b\in\{x,y,z\})$$

Its **trace is exactly $R_g^2$**, and diagonalising it gives three eigenvalues $\lambda_1\le\lambda_2\le\lambda_3$:

$$R_g^2=\operatorname{Tr}(S)=\lambda_1+\lambda_2+\lambda_3$$

The eigenvalues are the squared sizes along the molecule's three **principal axes**, so the tensor encodes the **shape**, not just the size: $\lambda_1\approx\lambda_2\approx\lambda_3$ → a sphere; one large $\lambda$ → a rod; one small $\lambda$ → a disk. The animation builds $S$ on a real V21 snapshot, diagonalises it, and draws the **gyration ellipsoid** (semi-axes $\sqrt{\lambda_i}$ along the eigenvectors), rotating so you can see how one $3\times3$ tensor captures the cloud's size, shape, and orientation at once.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from mpl_toolkits.mplot3d import Axes3D  # noqa: registers 3-D projection

xyz, box = load_coords("disordered", "dis", 0)
pts = residue_centroids(xyz, box)[600]                       # 21 residue centroids, one frame (nm)
com = pts.mean(0); dr = pts - com                            # displacement vectors from the COM
S = (dr[:, :, None] * dr[:, None, :]).mean(0)                # gyration tensor  S = <dr dr^T>
lam, vec = np.linalg.eigh(S)                                 # eigenvalues (asc) + principal axes
Rg = np.sqrt(lam.sum())                                      # Rg^2 = Tr(S) = sum of eigenvalues

# gyration ellipsoid: unit sphere scaled by sqrt(lambda) along the eigenvectors, centred at COM
a = np.linspace(0, 2*np.pi, 28); b = np.linspace(0, np.pi, 14)
unit = np.stack([np.outer(np.cos(a), np.sin(b)), np.outer(np.sin(a), np.sin(b)),
                 np.outer(np.ones_like(a), np.cos(b))], -1)
ell = (unit * np.sqrt(lam)) @ vec.T + com

fig = plt.figure(figsize=(6, 5)); ax = fig.add_subplot(111, projection="3d")
def upd(i):
    ax.clear()
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c="forestgreen", s=42)             # the beads
    ax.scatter([com[0]], [com[1]], [com[2]], c="black", s=70, marker="x")        # centre of mass
    ax.plot_surface(ell[...,0], ell[...,1], ell[...,2], color="orange", alpha=0.18, linewidth=0)
    ax.set_axis_off(); ax.view_init(elev=18, azim=i*6)
    ax.set_title(f"gyration ellipsoid    $R_g$ = {Rg:.2f} nm\n"
                 f"$\lambda$ = [{lam[0]:.2f}, {lam[1]:.2f}, {lam[2]:.2f}] nm$^2$   "
                 f"($R_g^2=\Sigma\lambda$ = {lam.sum():.2f})", fontsize=10)
    return ()
anim = animation.FuncAnimation(fig, upd, frames=60, interval=90)
plt.close(fig); HTML(anim.to_jshtml())

## Computing $R_g$ and end-to-end across the whole trajectory

Now the routine version: per frame, $R_g$ (the trace-of-tensor above) and the **end-to-end distance** (first-to-last nucleotide; we report $\sqrt{\langle R_{ee}^2\rangle}$). We unwrap across the periodic box first, and drop the first 10% as equilibration.

In [ ]:
def e2e_rg(variant, pre, salt, equil=0.1):
    xyz, box = load_coords(variant, pre, salt); c = residue_centroids(xyz, box)
    c = c[int(len(c)*equil):]
    e2e = np.linalg.norm(c[:, -1] - c[:, 0], axis=-1)
    rg  = np.sqrt(((c - c.mean(1, keepdims=True))**2).sum(-1).mean(-1))
    return e2e, rg
for v, pre in [("disordered","dis"), ("hairpin","hp")]:
    e, r = e2e_rg(v, pre, 150)
    print(f"{v:11s} 150 mM:  RMS e2e = {np.sqrt((e**2).mean()):.2f} nm   <Rg> = {r.mean():.2f} nm")

## The salt titration: dimensions vs salt

As salt rises, Debye screening weakens phosphate–phosphate repulsion, so the **disordered** chain **compacts**; the **hairpin** is held by restraints and barely moves.

In [ ]:
import matplotlib.pyplot as plt
SALTS = [0,10,20,50,100,150,200,300,400,500,700,1000]
fig, ax = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
for v, pre, c in [("disordered","dis","#1f77b4"), ("hairpin","hp","#d62728")]:
    rms = [np.sqrt((e2e_rg(v,pre,s)[0]**2).mean()) for s in SALTS]
    rg  = [e2e_rg(v,pre,s)[1].mean() for s in SALTS]
    ax[0].plot(SALTS, rms, "-o", color=c, label=v); ax[1].plot(SALTS, rg, "-o", color=c, label=v)
ax[0].set_ylabel("RMS end-to-end (nm)"); ax[1].set_ylabel("$R_g$ (nm)"); ax[1].set_xlabel("salt (mM)")
ax[0].legend(); ax[1].legend(); ax[0].set_title("V21 RNA: salt-induced compaction")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Temperature titration — and a lesson about model limits

The repo also ships 1 µs runs at **150 mM across 0–100 °C** (26 temperatures). Both forms **swell with temperature** (more thermal energy → larger, more expanded chains). But look closely at the hairpin: there's **no sharp melting transition** — it just breathes a little more. That's an honest statement about the *model*, not the RNA: our hairpin is held by **fixed-strength restraints that don't weaken with temperature**, so it physically *cannot* melt. A faithful melting curve would need temperature-dependent base-pairing (e.g. CALVADOS v0.8's structure-based restraints) — a great next step, and a reminder to always know what your model can and can't do.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
TEMPS = [0,5,10,15,16,20,23,25,30,35,37,40,44,45,50,55,56,60,65,70,75,80,85,90,95,100]
def e2e_rg_T(variant, pre, tC, equil=0.1):
    z = np.load(f"{DATA}/trajectories_temp/{variant}/V21_{pre}_{tC}C/coords.npz")
    c = residue_centroids(z["xyz"].astype(np.float64), z["box"].astype(np.float64))
    c = c[int(len(c)*equil):]
    rms = np.sqrt((np.linalg.norm(c[:, -1] - c[:, 0], axis=-1)**2).mean())
    rg  = np.sqrt(((c - c.mean(1, keepdims=True))**2).sum(-1).mean(-1)).mean()
    return rms, rg
fig, ax = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
for v, pre, col in [("disordered","dis","#1f77b4"), ("hairpin","hp","#d62728")]:
    vals = [e2e_rg_T(v, pre, t) for t in TEMPS]
    ax[0].plot(TEMPS, [x[0] for x in vals], "-o", ms=4, color=col, label=v)
    ax[1].plot(TEMPS, [x[1] for x in vals], "-o", ms=4, color=col, label=v)
ax[0].set_ylabel("RMS end-to-end (nm)"); ax[1].set_ylabel("$R_g$ (nm)")
ax[1].set_xlabel("temperature (°C)")
ax[0].legend(); ax[1].legend(); ax[0].set_title("V21 at 150 mM: dimensions vs temperature")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Free-energy profiles (PMF) — thermodynamics from the trajectory

A trajectory doesn't just give average sizes — it gives the **probability distribution** of a quantity, and from that a **free-energy profile** (potential of mean force):

$$F(x) = -k_BT\,\ln P(x) + \text{const}$$

The minimum is the most stable value; the curvature is how stiff it is; barriers are how hard it is to switch states. Below: $F(R_g)$ for the disordered and hairpin RNA at low vs high salt. For the **disordered** chain the well **shifts to smaller $R_g$** as salt rises (screening → compaction); the **hairpin** sits in a narrow, deep well that barely moves (the restraints dominate).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
kT = 0.008314 * 300            # kJ/mol at 300 K
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
for a, (v, pre) in zip(ax, [("disordered","dis"), ("hairpin","hp")]):
    for s, c in [(0,"#4575b4"), (150,"#999999"), (1000,"#d73027")]:
        _, r = e2e_rg(v, pre, s)
        h, ed = np.histogram(r, bins=45, density=True); mid = 0.5*(ed[1:]+ed[:-1]); m = h > 0
        F = -kT*np.log(h[m]); F -= F.min()
        a.plot(mid[m], F, "-", color=c, lw=2, label=f"{s} mM")
    a.set_title(f"{v}:  $F(R_g) = -k_BT\,\ln P(R_g)$")
    a.set_xlabel("$R_g$ (nm)"); a.set_ylabel("free energy (kJ/mol)")
    a.set_ylim(0, 8); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Contact maps — a threshold-based view of structure

A **contact map** asks, for every pair of residues $(i,j)$, *how often are they close?* We compute the per-frame residue–residue distance matrix, call a pair "in contact" when the distance is below a **threshold**, and average over the trajectory. The **hairpin** lights up an **anti-diagonal** (residue 3 near 19, 4 near 18, …) — the signature of a stem-loop — while the **disordered** chain shows only the trivial backbone diagonal. Change `THRESH` to see how the map depends on the contact cutoff.

In [ ]:
import matplotlib.pyplot as plt
THRESH = 1.2     # nm — the contact threshold (try 1.0, 1.5, ...)

def contact_freq(variant, pre, salt, thr, equil=0.1):
    xyz, box = load_coords(variant, pre, salt); c = residue_centroids(xyz, box)
    c = c[int(len(c)*equil):]
    d = np.linalg.norm(c[:, :, None, :] - c[:, None, :, :], axis=-1)   # (F,21,21)
    return (d < thr).mean(0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, (v, pre) in zip(ax, [("disordered","dis"), ("hairpin","hp")]):
    im = a.imshow(contact_freq(v, pre, 150, THRESH), origin="lower", cmap="viridis", vmin=0, vmax=1)
    a.set_title(f"{v}, 150 mM"); a.set_xlabel("residue $j$"); a.set_ylabel("residue $i$")
    for i, j in STEM_PAIRS:                       # mark the RNAfold stem pairs
        a.plot(j-1, i-1, "rs", mfc="none", ms=8); a.plot(i-1, j-1, "rs", mfc="none", ms=8)
    fig.colorbar(im, ax=a, fraction=0.046, label="contact frequency")
fig.suptitle(f"V21 contact maps (threshold {THRESH} nm; red squares = RNAfold stem pairs)")
plt.tight_layout(); plt.show()

## Contact-distance analysis (threshold-free)

Thresholds hide *how* close pairs are. Here we look at the actual distances: (left) the **mean residue–residue distance map** — the hairpin's stem shows up as a low-distance anti-diagonal with no threshold needed; (right) the **distance distributions of the 6 stem pairs** — pinned and narrow in the hairpin (restraints), broad and far in the disordered chain.

In [ ]:
import matplotlib.pyplot as plt
def mean_dist_map(variant, pre, salt, equil=0.1):
    xyz, box = load_coords(variant, pre, salt); c = residue_centroids(xyz, box)
    c = c[int(len(c)*equil):]
    return np.linalg.norm(c[:, :, None, :] - c[:, None, :, :], axis=-1).mean(0)
def stem_distances(variant, pre, salt, equil=0.1):
    xyz, box = load_coords(variant, pre, salt); c = residue_centroids(xyz, box)
    c = c[int(len(c)*equil):]
    return np.concatenate([np.linalg.norm(c[:, i-1] - c[:, j-1], axis=-1) for i, j in STEM_PAIRS])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
im = ax[0].imshow(mean_dist_map("hairpin","hp",150), origin="lower", cmap="magma_r")
ax[0].set_title("hairpin: mean residue–residue distance (nm)")
ax[0].set_xlabel("residue $j$"); ax[0].set_ylabel("residue $i$")
fig.colorbar(im, ax=ax[0], fraction=0.046, label="distance (nm)")
for v, pre, col in [("disordered","dis","#1f77b4"), ("hairpin","hp","#d62728")]:
    ax[1].hist(stem_distances(v, pre, 150), bins=50, density=True, alpha=0.6, color=col, label=v)
ax[1].set_xlabel("stem-pair residue distance (nm)"); ax[1].set_ylabel("density")
ax[1].legend(); ax[1].set_title("the 6 stem pairs: distance distribution")
plt.tight_layout(); plt.show()

## Watch it move — the trajectory in 3-D

Static snapshots only go so far. Here we **animate the uploaded trajectory** (py3Dmol) — the disordered chain wriggling through its conformations, and the hairpin holding its stem while the loop and tails flicker. Drag to rotate; it loops automatically.

In [ ]:
from IPython.display import display
print("disordered V21 (150 mM) — exploring many shapes:")
display(animate_traj("disordered", "dis", 150, nframes=60))
print("hairpin V21 (150 mM) — stem stays closed, ends fluctuate:")
display(animate_traj("hairpin", "hp", 150, nframes=60))

## See it: extended vs compact

The disordered chain at **0 mM** (extended) vs **1000 mM** (compact) — the salt effect, made visible.

In [ ]:
from IPython.display import display
print("disordered, 0 mM (extended):");   display(show_beads("disordered","dis",0,    frame=900))
print("disordered, 1000 mM (compact):"); display(show_beads("disordered","dis",1000, frame=900))

---
# Wrap-up

- **MD** integrates $\mathbf F=-\nabla U$; **coarse-graining** reaches long timescales; **CALVADOS** sums bond/angle/Ashbaugh–Hatch/Debye–Hückel terms.
- You **configured** a 2-bead CG RNA, saw it in 3-D, and imposed a **hairpin** with 6 restraints.
- You **analysed** 1 µs runs: **salt screens charge → disordered RNA compacts**, the **hairpin stays pinned**, and **contact maps** reveal the stem as an anti-diagonal.

**Next:** run your own (`platform='CUDA'`, `steps≈1e8`, on a GPU); try the temperature axis; a more faithful hairpin via CALVADOS v0.8's elastic-network restraints.

*CALVADOS: KULL-Centre/CALVADOS (Tesei et al.). Data + tutorial: this repository.*
